# Import

In [76]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader,Subset
from PIL import Image
import kagglehub

# Download Dataset

In [77]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


#setup

In [78]:
torch.manual_seed(42)
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(f" Using Device {device}")

 Using Device cuda


In [79]:
# #path (mistake)
# TRAIN_PATH = os.path.join(path,"Plantvillage","train")
# VAL_PATH = os.path.join(path,"Plantvillage","val")

In [80]:
TRAIN_PATH = os.path.join(path, "PlantVillage","train")
VAL_PATH = os.path.join(path, "PlantVillage","val")

# Transformation


In [81]:
transform = transforms.Compose(
    [
        transforms.Resize( (128,128) ),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

# Custom Dataset

In [82]:
from genericpath import isfile

class MultiClassClassfication(Dataset):
  def __init__(self, root_dir,transform=None) :
    super().__init__()

    self.samples = []
    self.transform = transform

    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

    self.class_to_idx= {cls_name : idx  for idx,cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)

      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path,img_name)

        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append( (img_path,label) )


  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx) :

    img_path, label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)


    return image, label



In [83]:
# example of enumerate

students = ['Alice', 'Bob', 'Charlie']
scores = [85, 92, 78]

# Using enumerate to get index and name
for idx, student in enumerate(students):
    print(f"{idx} → {student}, Score: {scores[idx]}")

0 → Alice, Score: 85
1 → Bob, Score: 92
2 → Charlie, Score: 78


# root/Apple___healthy/img1.jpg

# Example: {'Apple___healthy': 0, 'Tomato___late_blight': 1} → so label = 0 for 'Apple___healthy'

In [84]:
# [
#  ('root/Apple___healthy/img1.jpg', 0),
#  ('root/Apple___healthy/img2.jpg', 0),
#  ('root/Tomato___late_blight/img4.jpg', 1)
# ]

# <!-- Intuitive summary -->
# Outer loop → go through each class <br>
# Inner loop → go through each image in that class <br>
## Check → make sure it’s a file <br>
# Label → assign the numeric class <br>
# Store → save (image_path, label) for PyTorch <br>

# Load Data

In [85]:
# import os
# print(os.listdir("/kaggle/input/plantvillage"))

In [86]:
train_dataset_full = MultiClassClassfication(TRAIN_PATH, transform)
test_dataset_full = MultiClassClassfication(VAL_PATH, transform)
num_classes = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 38
Full train size: 43444
Full test size: 10861


#
# Use a subset for fast testing (optional)
#

In [87]:
train_dataset = Subset(train_dataset_full, list(range(min(100, len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full, list(range(min(100, len(test_dataset_full)))))


# DataLoader


In [88]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

# CNN

In [89]:
class MyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



In [90]:
model = MyCNN(num_classes=num_classes).to(device)


# Training Setup


In [91]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop


In [92]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 1.0934
Epoch 2/10, Loss: 0.0489
Epoch 3/10, Loss: 0.0003
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0000
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 0.0000


# Evaluation Function


In [93]:
def evaluate(loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return correct / total

In [94]:
test_acc = evaluate(test_loader)
print("Test Accuracy:", test_acc)

Test Accuracy: 1.0
